# ♟️ Benchmark 2 — Puzzles Lichess (réseau + recherche)

Le benchmark « 300 puzzles » des specs, en plus grand et **stratifié par difficulté** :
on tire des puzzles de la base officielle Lichess (~5 M puzzles notés) dans chaque tranche de classement,
et on mesure pour chaque *joueur* (réseau + recherche + profondeur) :

- **taux de résolution** (toute la séquence, un mat alternatif est accepté comme sur Lichess)
- **premier coup juste**
- **Elo puzzle estimé** : le classement de puzzle que le moteur résout 1 fois sur 2 (maximum de vraisemblance)
- **temps par coup**, résultats **par thème** (mat en 2, fourchette, clouage, finale…)
- **test de McNemar** entre modèles (même puzzles → écart significatif ou non)

Joueurs : `NOM@arbreN` = recherche arbre complet profondeur N (comme dans l'app), `PWN@abN` = poids SPAWN + alpha-bêta/quiescence profondeur N.

⚙️ Réglages Kaggle : **Internet ON**, **GPU T4** conseillé (la recherche arbre complet évalue ~1000 positions par coup en lot).
Les résultats sont sauvegardés au fur et à mesure : si la session coupe, relancer reprend où ça s'était arrêté.

In [ ]:
# onnxruntime-gpu remplace onnxruntime (les deux ne cohabitent pas) ; marche aussi sans GPU.
# Version fixée : les plus récentes demandent CUDA 13, Kaggle a CUDA 12. [cuda,cudnn] installe les bibliothèques CUDA 12.
!pip uninstall -y -q onnxruntime onnxruntime-gpu > /dev/null 2>&1
!pip install -q chess zstandard "onnxruntime-gpu[cuda,cudnn]==1.24.1"

In [ ]:
%%writefile commun_echecs.py
"""Code commun aux notebooks de benchmark échecs (PAWN, PAWN big, contrôle, SPAWN/PWN).

Tout vient de pawn_app.py : même encodage 69 octets, même recherche « arbre complet »,
même alpha-bêta + quiescence. Seule différence : pas de bruit ni de coups au hasard,
on veut mesurer la force réelle.
"""
import glob
import os
import shutil
import stat
import subprocess
import tarfile
import time
import urllib.request

import chess
import chess.engine
import chess.polyglot
import numpy as np
import onnxruntime as ort

DEPOT = "LiamLitle/ia-de-liam"
BRANCHE = "main"
LFS = f"https://media.githubusercontent.com/media/{DEPOT}/{BRANCHE}/"
DOSSIER_POIDS = os.environ.get("PAWN_POIDS", "/kaggle/working/poids" if os.path.isdir("/kaggle") else "poids")
MATE = 100_000

# nom -> chemin dans le dépôt
RESEAUX = {
    "PAWN": "Pawn/pawn.onnx",
    "PAWN_BIG": "PawnBig-V1/pawn_big.onnx",
    "CONTROLE": "PawnBig-controlleur/pawn_big_controle.onnx",
    "SPAWN": "PWN-soluce/pawn_soluce.onnx",
}


def poids(chemin_depot):
    """cherche le fichier dans /kaggle/input (dataset ajouté au notebook), sinon le télécharge depuis GitHub LFS"""
    nom = os.path.basename(chemin_depot)
    for racine in ("/kaggle/input", DOSSIER_POIDS):
        trouves = glob.glob(os.path.join(racine, "**", nom), recursive=True)
        trouves = [t for t in trouves if os.path.getsize(t) > 1000]  # pas un pointeur LFS
        if trouves:
            return trouves[0]
    os.makedirs(DOSSIER_POIDS, exist_ok=True)
    dest = os.path.join(DOSSIER_POIDS, nom)
    print(f"téléchargement de {chemin_depot} ...")
    urllib.request.urlretrieve(LFS + chemin_depot, dest)
    if os.path.getsize(dest) < 1000:
        raise RuntimeError(f"{dest} ressemble à un pointeur LFS, pas au vrai fichier")
    return dest


_CUDA_OK = None  # None = pas encore essayé, False = échec -> on reste sur CPU sans réessayer


def providers(gpu=True):
    dispo = ort.get_available_providers()
    if gpu and _CUDA_OK is not False and "CUDAExecutionProvider" in dispo:
        try:
            ort.preload_dlls()  # charge CUDA/cuDNN depuis les paquets pip nvidia-* (déjà là avec torch sur Kaggle)
        except Exception:
            pass
        return ["CUDAExecutionProvider", "CPUExecutionProvider"]
    return ["CPUExecutionProvider"]


PIECES_ID = {c: i + 1 for i, c in enumerate("PNBRQK")}


def enc_fen(fen):
    f = fen.split(" ")
    noir = f[1] == "b"
    out = bytearray(69)
    r = c = 0
    for ch in f[0]:
        if ch == "/":
            r += 1
            c = 0
        elif ch.isdigit():
            c += int(ch)
        else:
            nous = ch.isupper() != noir
            out[(r if noir else 7 - r) * 8 + c] = PIECES_ID[ch.upper()] + (0 if nous else 6)
            c += 1
    ours, theirs = ("kq", "KQ") if noir else ("KQ", "kq")
    out[64] = ours[0] in f[2]
    out[65] = ours[1] in f[2]
    out[66] = theirs[0] in f[2]
    out[67] = theirs[1] in f[2]
    if f[3] != "-":
        out[68] = ord(f[3][0]) - 96
    return bytes(out)


class Evaluateur:
    """un réseau ONNX : liste de FEN -> centipions du point de vue du camp au trait"""

    def __init__(self, nom, gpu=True, threads=0):
        self.nom = nom
        so = ort.SessionOptions()
        if threads:
            so.intra_op_num_threads = threads
            so.inter_op_num_threads = 1
        global _CUDA_OK
        prov = providers(gpu)
        self.session = ort.InferenceSession(poids(RESEAUX[nom]), so, providers=prov)
        if "CUDAExecutionProvider" in prov:
            _CUDA_OK = "CUDAExecutionProvider" in self.session.get_providers()
            if not _CUDA_OK:
                print("⚠️ GPU indisponible pour onnxruntime : on continue sur CPU (plus lent mais résultats identiques)")
                ort.set_default_logger_severity(4)
        self.nb_evals = 0
        self.cache = {}

    def evals(self, fens, lot=4096):
        if not fens:
            return np.zeros(0, dtype=np.float32)
        res = []
        for i in range(0, len(fens), lot):
            x = np.frombuffer(b"".join(map(enc_fen, fens[i:i + lot])), np.uint8).reshape(-1, 69).copy()
            res.append(self.session.run(None, {"board": x})[0])
        self.nb_evals += len(fens)
        return np.concatenate(res)

    def eval1(self, fen):
        # l'alpha-bêta réévalue souvent les mêmes positions en quiescence : le réseau est
        # déterministe, donc un cache ne change rien au résultat, juste la vitesse
        v = self.cache.get(fen)
        if v is None:
            if len(self.cache) > 2_000_000:
                self.cache.clear()
            v = self.cache[fen] = float(self.evals([fen])[0])
        return v


# -- recherche « arbre complet » (PAWN, PAWN big, contrôle, SPAWN) -----------------

def arbre(board, depth, ply, feuilles):
    if not any(board.legal_moves):
        return ("t", -(MATE - ply) if board.is_check() else 0)
    if ply > 0 and (board.is_insufficient_material() or board.is_repetition(2) or board.halfmove_clock >= 100):
        return ("t", 0)
    if depth == 0:
        feuilles.append(board.fen())
        return ("f", len(feuilles) - 1)
    fils = []
    for mv in list(board.legal_moves):
        board.push(mv)
        fils.append((mv, arbre(board, depth - 1, ply + 1, feuilles)))
        board.pop()
    return ("n", fils)


def valeur(noeud, v):
    genre, x = noeud
    if genre == "t":
        return x
    if genre == "f":
        return v[x]
    return max(-valeur(c, v) for _, c in x)


def choisir_arbre(board, ev, depth):
    feuilles = []
    racine = arbre(board, depth, 0, feuilles)
    v = ev.evals(feuilles) if feuilles else []
    notes = [(-valeur(c, v), mv) for mv, c in racine[1]]
    return max(notes, key=lambda t: t[0])[1]


# -- recherche alpha-bêta + quiescence (PWN) --------------------------------------

VAL_PIECE = {chess.PAWN: 1, chess.KNIGHT: 3, chess.BISHOP: 3, chess.ROOK: 5, chess.QUEEN: 9, chess.KING: 0}


def trier_coups(board, coups, priorite):
    def cle(mv):
        if mv == priorite:
            return 10_000
        if board.is_capture(mv):
            prise = board.piece_type_at(mv.to_square) or chess.PAWN
            attaquant = board.piece_type_at(mv.from_square)
            return 1000 + VAL_PIECE[prise] * 10 - VAL_PIECE[attaquant]
        if board.gives_check(mv):
            return 500
        return 0
    return sorted(coups, key=cle, reverse=True)


def quiescence(board, ev, alpha, beta, prof_q):
    stand_pat = ev.eval1(board.fen())
    if stand_pat >= beta:
        return beta
    alpha = max(alpha, stand_pat)
    if prof_q <= 0:
        return alpha
    captures = [m for m in board.legal_moves if board.is_capture(m)]
    for mv in trier_coups(board, captures, None):
        board.push(mv)
        score = -quiescence(board, ev, -beta, -alpha, prof_q - 1)
        board.pop()
        if score >= beta:
            return beta
        alpha = max(alpha, score)
    return alpha


def negamax(board, ev, profondeur, alpha, beta, ply, prof_q, tt, coup_tt):
    alpha0 = alpha
    if not any(board.legal_moves):
        return -(MATE - ply) if board.is_check() else 0
    if ply > 0 and (board.is_insufficient_material() or board.is_repetition(2) or board.halfmove_clock >= 100):
        return 0
    hash_ = chess.polyglot.zobrist_hash(board)
    entree = tt.get(hash_)
    if entree and entree[0] >= profondeur:
        d, v, drapeau, _ = entree
        if drapeau == "exact":
            return v
        if drapeau == "min":
            alpha = max(alpha, v)
        elif drapeau == "max":
            beta = min(beta, v)
        if alpha >= beta:
            return v
    if profondeur <= 0:
        return quiescence(board, ev, alpha, beta, prof_q)
    meilleur, meilleur_coup = -MATE - 1, None
    for mv in trier_coups(board, list(board.legal_moves), coup_tt.get(hash_)):
        board.push(mv)
        score = -negamax(board, ev, profondeur - 1, -beta, -alpha, ply + 1, prof_q, tt, coup_tt)
        board.pop()
        if score > meilleur:
            meilleur, meilleur_coup = score, mv
        alpha = max(alpha, score)
        if alpha >= beta:
            break
    drapeau = "exact" if alpha0 < meilleur < beta else ("min" if meilleur >= beta else "max")
    tt[hash_] = (profondeur, meilleur, drapeau, meilleur_coup)
    if meilleur_coup is not None:
        coup_tt[hash_] = meilleur_coup
    return meilleur


def choisir_alphabeta(board, ev, depth, prof_q=4):
    tt, coup_tt = {}, {}
    meilleur_coup = None
    for d in range(1, depth + 1):
        alpha, beta = -MATE - 1, MATE + 1
        hash_ = chess.polyglot.zobrist_hash(board)
        meilleur_score, meilleur_du_tour = -MATE - 1, None
        for mv in trier_coups(board, list(board.legal_moves), coup_tt.get(hash_)):
            board.push(mv)
            score = -negamax(board, ev, d - 1, -beta, -alpha, 1, prof_q, tt, coup_tt)
            board.pop()
            if score > meilleur_score:
                meilleur_score, meilleur_du_tour = score, mv
            alpha = max(alpha, score)
        meilleur_coup = meilleur_du_tour
        coup_tt[hash_] = meilleur_coup
    return meilleur_coup


# -- joueurs ------------------------------------------------------------------------
# un « joueur » = un réseau + un algorithme de recherche + une profondeur.
# ex : "PAWN_BIG@arbre2", "SPAWN@arbre1", "PWN@ab3" (PWN = poids SPAWN + alpha-bêta).

def decoder(joueur):
    nom, rech = joueur.split("@")
    if nom == "PWN":
        nom = "SPAWN"
    if rech.startswith("arbre"):
        return nom, "arbre", int(rech[5:])
    if rech.startswith("ab"):
        return nom, "ab", int(rech[2:])
    raise ValueError(joueur)


_EVALS = {}
_SF = {}
GPU = True
THREADS = 0
SF_CHEMIN = None
SF_TEMPS = 0.1


def evaluateur(nom):
    if nom not in _EVALS:
        _EVALS[nom] = Evaluateur(nom, gpu=GPU, threads=THREADS)
    return _EVALS[nom]


def configurer(gpu=True, threads=0, sf_chemin=None, sf_temps=0.1):
    """à appeler dans chaque processus fils avant de jouer"""
    global GPU, THREADS, SF_CHEMIN, SF_TEMPS
    GPU, THREADS, SF_CHEMIN, SF_TEMPS = gpu, threads, sf_chemin, sf_temps
    _EVALS.clear()
    _SF.clear()


def stockfish(cfg):
    """cfg = "elo1500" (UCI_LimitStrength) ou "skill0" (Skill Level)"""
    if cfg not in _SF:
        e = chess.engine.SimpleEngine.popen_uci(SF_CHEMIN)
        opts = {"Threads": 1, "Hash": 16}
        if cfg.startswith("elo"):
            opts.update({"UCI_LimitStrength": True, "UCI_Elo": int(cfg[3:])})
        elif cfg.startswith("skill"):
            opts["Skill Level"] = int(cfg[5:])
        e.configure(opts)
        _SF[cfg] = e
    return _SF[cfg]


def fermer_stockfish():
    for e in _SF.values():
        e.quit()
    _SF.clear()


def coup(joueur, board):
    if joueur.startswith("SF@"):
        return stockfish(joueur[3:]).play(board, chess.engine.Limit(time=SF_TEMPS)).move
    nom, rech, depth = decoder(joueur)
    ev = evaluateur(nom)
    b = board.copy(stack=True)
    if rech == "arbre":
        return choisir_arbre(b, ev, depth)
    return choisir_alphabeta(b, ev, depth)


# -- Stockfish -------------------------------------------------------------------------

URLS_STOCKFISH = [
    "https://github.com/official-stockfish/Stockfish/releases/download/sf_17.1/stockfish-ubuntu-x86-64-avx2.tar",
    "https://github.com/official-stockfish/Stockfish/releases/download/sf_17/stockfish-ubuntu-x86-64-avx2.tar",
    "https://github.com/official-stockfish/Stockfish/releases/download/sf_16.1/stockfish-ubuntu-x86-64-avx2.tar",
]


def installer_stockfish(dossier=None):
    dossier = dossier or os.path.join(os.path.dirname(os.path.abspath(DOSSIER_POIDS)), "stockfish")
    deja = glob.glob(os.path.join(dossier, "**", "stockfish-ubuntu-*"), recursive=True)
    deja = [d for d in deja if os.path.isfile(d) and not d.endswith(".tar")]
    if deja:
        return deja[0]
    os.makedirs(dossier, exist_ok=True)
    for url in URLS_STOCKFISH:
        try:
            tar = os.path.join(dossier, "sf.tar")
            urllib.request.urlretrieve(url, tar)
            with tarfile.open(tar) as t:
                t.extractall(dossier)
            binaire = [d for d in glob.glob(os.path.join(dossier, "**", "stockfish-ubuntu-*"), recursive=True)
                       if os.path.isfile(d) and not d.endswith(".tar")][0]
            os.chmod(binaire, os.stat(binaire).st_mode | stat.S_IEXEC)
            subprocess.run([binaire, "quit"], check=True, timeout=10)
            return binaire
        except Exception as e:
            print("échec", url, e)
    if shutil.which("stockfish") is None:
        subprocess.run("apt-get -qq install -y stockfish", shell=True)
    for p in (shutil.which("stockfish"), "/usr/games/stockfish"):
        if p and os.path.exists(p):
            return p
    raise RuntimeError("impossible d'installer Stockfish")


# -- puzzles Lichess ---------------------------------------------------------------

def resoudre_puzzle(joueur, fen, moves):
    """format Lichess : FEN avant le coup adverse, moves[0] = coup adverse, puis solution.
    Comme sur Lichess, un mat est toujours accepté même si ce n'est pas le coup attendu.
    renvoie (résolu, premier_coup_bon, nb_coups_joués, secondes)"""
    b = chess.Board(fen)
    moves = moves.split()
    b.push_uci(moves[0])
    premier, joues, t0 = None, 0, time.perf_counter()
    for i in range(1, len(moves), 2):
        attendu = chess.Move.from_uci(moves[i])
        mv = coup(joueur, b)
        joues += 1
        b.push(mv)
        mat = b.is_checkmate()
        b.pop()
        ok = mv == attendu or mat
        if premier is None:
            premier = ok
        if not ok:
            return False, premier, joues, time.perf_counter() - t0
        if mat:
            break
        b.push(attendu)
        if i + 1 < len(moves):
            b.push_uci(moves[i + 1])
    return True, premier, joues, time.perf_counter() - t0


# -- parties -------------------------------------------------------------------------

def jouer_partie(blancs, noirs, ouverture, max_plies=300):
    """ouverture = liste de coups UCI joués d'office ; renvoie un dict (résultat, pgn, temps)"""
    import chess.pgn
    b = chess.Board()
    for u in ouverture:
        b.push_uci(u)
    temps = {blancs: 0.0, noirs: 0.0}
    nb = {blancs: 0, noirs: 0}
    while not b.is_game_over(claim_draw=True) and b.ply() < max_plies:
        j = blancs if b.turn == chess.WHITE else noirs
        t0 = time.perf_counter()
        mv = coup(j, b)
        temps[j] += time.perf_counter() - t0
        nb[j] += 1
        b.push(mv)
    res = b.result(claim_draw=True) if b.is_game_over(claim_draw=True) else "1/2-1/2"
    fin = b.outcome(claim_draw=True)
    jeu = chess.pgn.Game.from_board(b)
    jeu.headers.update(Event="Benchmark P.A.W.N.", White=blancs, Black=noirs, Result=res)
    return {
        "blancs": blancs, "noirs": noirs, "resultat": res,
        "fin": fin.termination.name if fin else "MAX_PLIES",
        "plies": b.ply(), "pgn": str(jeu),
        "s_par_coup_blancs": temps[blancs] / max(1, nb[blancs]),
        "s_par_coup_noirs": temps[noirs] / max(1, nb[noirs]),
    }


# -- tâches pour ProcessPoolExecutor (doivent être importables depuis ce module) -------

def init_worker(gpu, threads, sf_chemin=None, sf_temps=0.1):
    configurer(gpu=gpu, threads=threads, sf_chemin=sf_chemin, sf_temps=sf_temps)


def tache_puzzle(args):
    joueur, pid, fen, moves = args
    ok, premier, joues, sec = resoudre_puzzle(joueur, fen, moves)
    return {"joueur": joueur, "PuzzleId": pid, "resolu": ok, "premier_coup": premier,
            "coups_joues": joues, "secondes": sec}


def tache_partie(args):
    blancs, noirs, ouverture, max_plies, id_ouv = args
    try:
        r = jouer_partie(blancs, noirs, ouverture, max_plies)
    finally:
        # un Stockfish ouvert garde un thread vivant qui empêche le processus fils de se terminer
        fermer_stockfish()
    r["ouverture"] = id_ouv
    return r

In [ ]:
import os, sys, time, json, random
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import chess
sys.path.insert(0, os.getcwd())  # pour que les processus fils trouvent commun_echecs.py
import onnxruntime as ort
import commun_echecs as ce

print("onnxruntime", ort.__version__, "| providers dispo :", ort.get_available_providers())
ev_test = ce.Evaluateur("PAWN")
print("session PAWN sur :", ev_test.session.get_providers())
print("éval position initiale :", ev_test.eval1(chess.STARTING_FEN), "cp")

In [ ]:
# ---- réglages ----
TRANCHES = list(range(400, 2801, 200))   # bornes des tranches de classement puzzle
PAR_TRANCHE = 100                         # puzzles par tranche (12 tranches -> 1200 puzzles)
PAR_TRANCHE_LENT = 40                     # pour les joueurs lents (PWN@ab3)
JOUEURS = [
    "PAWN@arbre1", "PAWN_BIG@arbre1", "CONTROLE@arbre1", "SPAWN@arbre1",
    "PAWN@arbre2", "PAWN_BIG@arbre2", "CONTROLE@arbre2", "SPAWN@arbre2",
    "PWN@ab2", "PWN@ab3",
]
LENTS = {"PWN@ab3"}
GPU = True
NB_WORKERS = os.cpu_count()
THREADS = 1                                # threads onnxruntime par processus (CPU)
SEED = 0
SORTIE = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."

## 1. Base de puzzles Lichess

In [ ]:
import io, urllib.request
FICHIER = f"{SORTIE}/lichess_db_puzzle.csv.zst"
COLS = ["PuzzleId", "FEN", "Moves", "Rating", "RatingDeviation", "Popularity", "NbPlays", "Themes"]
try:
    import zstandard
    if not os.path.exists(FICHIER):
        urllib.request.urlretrieve("https://database.lichess.org/lichess_db_puzzle.csv.zst", FICHIER)
    with open(FICHIER, "rb") as fh:
        texte = io.TextIOWrapper(zstandard.ZstdDecompressor().stream_reader(fh), encoding="utf-8")
        puzzles = pd.read_csv(texte, usecols=COLS)
except Exception as e:
    print("database.lichess.org indisponible :", e, "-> Hugging Face")
    from datasets import load_dataset
    puzzles = load_dataset("Lichess/chess-puzzles", split="train").to_pandas()[COLS]
print(f"{len(puzzles):,} puzzles")

In [ ]:
# puzzles fiables : classement stable, bien notés, beaucoup joués
ok = puzzles[(puzzles.RatingDeviation < 90) & (puzzles.Popularity > 80) & (puzzles.NbPlays > 1000)].copy()
ok["tranche"] = pd.cut(ok.Rating, TRANCHES, right=False)
ok = ok.dropna(subset=["tranche"])
ok = ok.sample(frac=1, random_state=SEED)  # mélange, puis les PAR_TRANCHE premiers de chaque tranche
echant = ok[ok.groupby("tranche", observed=True).cumcount() < PAR_TRANCHE].sort_values("Rating").reset_index(drop=True)
# sous-échantillon pour les joueurs lents : les PAR_TRANCHE_LENT premiers de chaque tranche
echant["lent"] = echant.groupby("tranche", observed=True).cumcount() < PAR_TRANCHE_LENT
echant.to_csv(f"{SORTIE}/bench2_puzzles_echantillon.csv", index=False)
print(len(echant), "puzzles,", echant.lent.sum(), "pour les joueurs lents")
echant.groupby("tranche", observed=True).size()

## 2. Résolution (en parallèle, avec reprise)

In [ ]:
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor
from tqdm.auto import tqdm

RESULTATS = f"{SORTIE}/bench2_resultats_bruts.csv"
deja = pd.read_csv(RESULTATS) if os.path.exists(RESULTATS) else pd.DataFrame(columns=["joueur", "PuzzleId"])
fait = set(zip(deja.joueur, deja.PuzzleId))

for joueur in JOUEURS:
    lot = echant[echant.lent] if joueur in LENTS else echant
    taches = [(joueur, r.PuzzleId, r.FEN, r.Moves) for r in lot.itertuples() if (joueur, r.PuzzleId) not in fait]
    if not taches:
        print(joueur, ": déjà fait"); continue
    t0 = time.time()
    with ProcessPoolExecutor(NB_WORKERS, mp_context=mp.get_context("spawn"),
                             initializer=ce.init_worker, initargs=(GPU, THREADS)) as ex:
        tampon = []
        for r in tqdm(ex.map(ce.tache_puzzle, taches, chunksize=2), total=len(taches), desc=joueur):
            tampon.append(r)
            if len(tampon) >= 50:
                pd.DataFrame(tampon).to_csv(RESULTATS, mode="a", header=not os.path.exists(RESULTATS), index=False)
                tampon = []
        if tampon:
            pd.DataFrame(tampon).to_csv(RESULTATS, mode="a", header=not os.path.exists(RESULTATS), index=False)
    print(f"{joueur} : {len(taches)} puzzles en {(time.time() - t0) / 60:.1f} min")

## 3. Résultats

In [ ]:
from scipy.optimize import minimize_scalar

res = pd.read_csv(RESULTATS).drop_duplicates(["joueur", "PuzzleId"]).merge(echant, on="PuzzleId")
res["resolu"] = res["resolu"].astype(bool); res["premier_coup"] = res["premier_coup"].astype(bool)

def elo_puzzle(ratings, ok):
    """classement E tel que P(résolu) = 1 / (1 + 10^((R - E) / 400)) colle le mieux aux résultats"""
    r, y = np.asarray(ratings, float), np.asarray(ok, float)
    def nll(e):
        p = np.clip(1 / (1 + 10 ** ((r - e) / 400)), 1e-9, 1 - 1e-9)
        return -np.sum(y * np.log(p) + (1 - y) * np.log(1 - p))
    return minimize_scalar(nll, bounds=(-500, 4000), method="bounded").x

def resume(g):
    return pd.Series({
        "puzzles": len(g),
        "résolus (%)": 100 * g.resolu.mean(),
        "premier coup (%)": 100 * g.premier_coup.mean(),
        "Elo puzzle": elo_puzzle(g.Rating, g.resolu),
        "s / coup": g.secondes.sum() / g.coups_joues.sum(),
    })

tableau = res.groupby("joueur").apply(resume).loc[[j for j in JOUEURS if j in set(res.joueur)]]
# même comparaison sur le sous-échantillon commun à tous les joueurs (utile pour PWN@ab3)
tableau["Elo puzzle (sous-éch. commun)"] = res[res.lent].groupby("joueur").apply(lambda g: elo_puzzle(g.Rating, g.resolu))
tableau.to_csv(f"{SORTIE}/bench2_tableau.csv")
tableau.round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
par_tranche = res.groupby(["joueur", "tranche"], observed=True).resolu.mean().unstack(0) * 100
par_tranche.index = [str(i.left) if hasattr(i, "left") else str(i).split(",")[0].strip("[") for i in par_tranche.index]
par_tranche[[j for j in JOUEURS if j in par_tranche]].plot(ax=axes[0], marker="o")
axes[0].set(xlabel="classement du puzzle", ylabel="résolus (%)", title="Taux de résolution par difficulté")
axes[0].grid(alpha=.3)
tableau["Elo puzzle"].plot.barh(ax=axes[1], color="#58a", title="Elo puzzle estimé")
for i, v in enumerate(tableau["Elo puzzle"]):
    axes[1].text(v, i, f" {v:.0f}", va="center")
plt.tight_layout(); plt.savefig(f"{SORTIE}/bench2_puzzles.png", dpi=120); plt.show()

In [ ]:
THEMES = ["mateIn1", "mateIn2", "mateIn3", "fork", "pin", "skewer", "discoveredAttack", "hangingPiece",
          "sacrifice", "deflection", "attraction", "kingsideAttack", "defensiveMove", "quietMove",
          "opening", "middlegame", "endgame", "pawnEndgame", "rookEndgame", "short", "long", "veryLong"]
lignes = []
for t in THEMES:
    g = res[res.Themes.str.split().apply(lambda l: t in l)]
    if len(g):
        s = g.groupby("joueur").resolu.mean() * 100
        s["n (par joueur)"] = g.groupby("joueur").size().max()
        s.name = t
        lignes.append(s)
themes = pd.DataFrame(lignes)[[j for j in JOUEURS if j in set(res.joueur)] + ["n (par joueur)"]]
themes.to_csv(f"{SORTIE}/bench2_themes.csv")
themes.style.format("{:.0f}").background_gradient(axis=1, cmap="RdYlGn", subset=themes.columns[:-1])

### Tests de McNemar
Sur les mêmes puzzles : combien A résout que B rate (et inversement), et est-ce significatif ?

In [ ]:
from scipy.stats import binomtest
PAIRES = [("PAWN@arbre2", "PAWN_BIG@arbre2"), ("PAWN_BIG@arbre2", "CONTROLE@arbre2"),
          ("CONTROLE@arbre2", "SPAWN@arbre2"), ("SPAWN@arbre2", "PWN@ab2"), ("PWN@ab2", "PWN@ab3"),
          ("CONTROLE@arbre1", "SPAWN@arbre1")]
piv = res.pivot_table(index="PuzzleId", columns="joueur", values="resolu")
lignes = []
for a, b in PAIRES:
    if a in piv and b in piv:
        x = piv[[a, b]].dropna().astype(bool)
        seul_b, seul_a = int((~x[a] & x[b]).sum()), int((x[a] & ~x[b]).sum())
        p = binomtest(seul_b, seul_a + seul_b).pvalue if seul_a + seul_b else 1.0
        lignes.append({"A": a, "B": b, "puzzles": len(x), "B résout, pas A": seul_b, "A résout, pas B": seul_a,
                       "p-value": p, "significatif (5 %)": p < 0.05})
mcnemar = pd.DataFrame(lignes); mcnemar.to_csv(f"{SORTIE}/bench2_mcnemar.csv", index=False)
mcnemar

### Comment lire les résultats
- L'**Elo puzzle** est le chiffre le plus parlant : il est sur l'échelle des puzzles Lichess (pas l'Elo de partie !).
- **CONTROLE vs SPAWN** à profondeur égale isole l'apport de Soluce ; **SPAWN@arbre2 vs PWN@ab2** isole l'apport de la recherche alpha-bêta/quiescence.
- Les anciens chiffres (300 puzzles) ne sont pas directement comparables : ici la difficulté est répartie uniformément de 400 à 2800.